In [0]:
dbutils.widgets.text("catalog","default")
catalog=dbutils.widgets.get("catalog")

In [0]:
spark.sql(f"CREATE TABLE IF NOT EXISTS {catalog}.bronze.stations USING delta COMMENT 'VoltPath stations csv files'")
spark.sql(f"""
COPY INTO {catalog}.bronze.stations
FROM (
    SELECT *, _metadata.file_path AS _source_file, current_timestamp() AS _ingested_at
    FROM '/Volumes/vp_dp20040222/landing/raw/stations/'
)
FILEFORMAT = csv
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true')
""")

In [0]:
spark.sql(f"CREATE TABLE IF NOT EXISTS {catalog}.bronze.tariffs USING delta COMMENT 'VoltPath tariffs csv files'")
spark.sql(f"""
COPY INTO {catalog}.bronze.tariffs
FROM (
    SELECT *, _metadata.file_path AS _source_file, current_timestamp() AS _ingested_at
    FROM '/Volumes/vp_dp20040222/landing/raw/tariffs/'
)
FILEFORMAT = csv
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true')
""")

In [0]:
spark.sql(f"CREATE TABLE IF NOT EXISTS {catalog}.bronze.members USING delta COMMENT 'VoltPath members csv files'")
spark.sql(f"""
COPY INTO {catalog}.bronze.members
FROM (
    SELECT *, _metadata.file_path AS _source_file, current_timestamp() AS _ingested_at, CAST(NULL AS TIMESTAMP) AS change_effective_date
    FROM '/Volumes/vp_dp20040222/landing/raw/members/'
)
FILEFORMAT = csv
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true')
""")

In [0]:
spark.sql(f"CREATE TABLE IF NOT EXISTS {catalog}.bronze.chargers USING delta COMMENT 'raw chargers json files'")
spark.sql(f"""
COPY INTO {catalog}.bronze.chargers
  FROM '/Volumes/{catalog}/landing/raw/chargers/'
  FILEFORMAT = JSON
  FILES = ('chargers_2026-04-01.json')
  FORMAT_OPTIONS ('multiLine' = 'true')
  COPY_OPTIONS ('mergeSchema' = 'true')
""")

In [0]:
# Create the target Bronze table
source_path = f"/Volumes/{catalog}/landing/raw/sessions/"
target_table = f"{catalog}.bronze.sessions"
checkpoint_path = f"/Volumes/{catalog}/ops/pipeline_state/checkpoints/sessions"
schema_path = f"/Volumes/{catalog}/ops/pipeline_state/schema/sessions"

In [0]:
# Read files 
df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")
    .load(source_path)
)

In [0]:
# Add metadata columns for file path and ingestion timestamp
from pyspark.sql.functions import col, current_timestamp

df_stream = (
    df_stream
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
)

# Write to Bronze Delta table
query = (
    df_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

In [0]:
%sql
select count(*) from vp_dp20040222.bronze.stations;

In [0]:
%sql
select count(*) from vp_dp20040222.bronze.tariffs;

In [0]:
%sql
select count(*) from vp_dp20040222.bronze.members;

In [0]:
%sql
select count(*) from vp_dp20040222.bronze.chargers;

In [0]:
%sql
select count(*) from vp_dp20040222.bronze.sessions;